# 拓扑序与电荷序共存模型的 QGN 分析

本 notebook 分析来自综述《拓扑序与对称破缺序的共存》的三个关键模型，通过量子几何嵌套（QGN）框架诊断其 CDW 不稳定性。

## 模型列表

| 编号 | 模型 | 晶格 | 参考文献 | 综述章节 |
|------|------|------|---------|----------|
| A | Haldane 蜂窝晶格 Chern 绝缘体 | 蜂窝 | Haldane (1988); Neupert et al. (2011) | §3.2 |
| B | QWZ 方格晶格 Chern 绝缘体 | 正方 | Qi–Wu–Zhang (2006); Tešanović (1989) | §2.3 |
| C | 三角晶格 FCI | 三角 | Kourtis–Venderbos–Daghofer (2012) | §4.1 |

模型 A 是 Chern 绝缘体的标准参考（非平带）。模型 B 是最简单的方格晶格拓扑绝缘体，与综述 §2.3 Tešanović 磁通相绝缘体同属一类。模型 C 是 FCI 近平带模型，在强关联下展现 FCI×CDW 共存。QGN 嵌套图（nestability map）揭示哪个波矢 **Q** 处几何嵌套最强——间接反映相互作用驱动下的 CDW 倾向。

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from qgn.models import (haldane_hamiltonian, qwz_hamiltonian,
                         fci_triangular_hamiltonian,
                         HAL_B1, HAL_B2, QWZ_B1, QWZ_B2)
from qgn.geometry import (diagonalize_model, projection_matrix_from_vecs,
                           berry_curvature_grid, chern_number, quantum_distance)
from qgn.core import nestability_map, nesting_matrix

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('imports OK')

## 2. 模型 A — Haldane 蜂窝晶格

**哈密顿量**（Haldane 1988）：
$$H(\mathbf{k}) = \begin{pmatrix} M + 2t_2\sum_i\cos(\mathbf{k}\cdot\mathbf{v}_i+\varphi) & t_1\sum_j e^{i\mathbf{k}\cdot\boldsymbol{\delta}_j} \\ \text{h.c.} & -M + 2t_2\sum_i\cos(\mathbf{k}\cdot\mathbf{v}_i-\varphi) \end{pmatrix}$$

拓扑相（$C=+1$）条件：$|M| < 3\sqrt{3}\,t_2|\sin\varphi|$。取 $M=0,\,\varphi=\pi/2$：对任意 $t_2\neq0$ 均处于拓扑相。

Neupert et al. (2011) 用此模型（加近邻相互作用 $V$）在 $\nu=1/3$ 处发现 FCI。

In [ ]:
# ── Haldane parameters ──────────────────────────────────────────────────────
t1_H, t2_H, phi_H, M_H = 1.0, 0.3, np.pi/2, 0.0

# High-symmetry points (Cartesian) in the honeycomb BZ
Gamma_H = np.array([0.0, 0.0])
K_H     = (2*HAL_B1 + HAL_B2) / 3          # K = (2b1+b2)/3
Kp_H    = (HAL_B1 + 2*HAL_B2) / 3          # K'
M_sym_H = HAL_B1 / 2                        # M midpoint

def make_path(pts, N=80):
    segs = [np.outer(np.linspace(0,1,N,endpoint=False), p1-p0)+p0
            for p0,p1 in zip(pts[:-1],pts[1:])]
    segs.append(pts[-1:])
    return np.vstack(segs)

k_path_H  = make_path([Gamma_H, M_sym_H, K_H, Gamma_H])
k_dist_H  = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(k_path_H, axis=0), axis=1))])
tick_pos_H = [0,
              np.linalg.norm(M_sym_H),
              np.linalg.norm(M_sym_H) + np.linalg.norm(K_H - M_sym_H),
              k_dist_H[-1]]

# Band structure for different t2/t1
fig, ax = plt.subplots(figsize=(6, 4))
colors = plt.cm.viridis(np.linspace(0.1, 0.85, 4))
for i, t2_v in enumerate([0.0, 0.1, 0.3, 0.5]):
    Es = np.array([np.linalg.eigvalsh(haldane_hamiltonian(k, t1=t1_H, t2=t2_v, phi=phi_H))
                   for k in k_path_H])
    for n in range(2):
        ax.plot(k_dist_H, Es[:, n], color=colors[i],
                lw=1.5, label=f"$t_2/t_1={t2_v}$" if n == 0 else None)

for xp in tick_pos_H:
    ax.axvline(xp, color='gray', lw=0.5, ls='--')
ax.set_xticks(tick_pos_H)
ax.set_xticklabels(['Γ', 'M', 'K', 'Γ'])
ax.set_ylabel('$E/t_1$')
ax.set_title('Haldane Honeycomb Band Structure  ($M=0,\,\\varphi=\\pi/2$)')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/haldane_bands.png', dpi=110)
plt.show()

# Bandwidth and gap for t2=0.3
Es_default = np.array([np.linalg.eigvalsh(haldane_hamiltonian(k, t1=t1_H, t2=t2_H, phi=phi_H))
                        for k in k_path_H])
W_H   = Es_default[:, 0].max() - Es_default[:, 0].min()
gap_H = Es_default[:, 1].min() - Es_default[:, 0].max()
print(f't2/t1 = {t2_H}: lower-band width W = {W_H:.4f},  gap Δ = {gap_H:.4f},  M = Δ/W = {gap_H/W_H:.2f}')
print(f'Topological phase condition: |M| < 3√3·t2·|sinφ| = {3*np.sqrt(3)*t2_H*abs(np.sin(phi_H)):.4f}')

In [ ]:
# ── Berry curvature & Chern number (Haldane) ────────────────────────────────
Nk = 40
k1v = np.linspace(0, 1, Nk, endpoint=False)
k2v = np.linspace(0, 1, Nk, endpoint=False)

evecs_H_2d = np.zeros((Nk, Nk, 2, 2), dtype=complex)
K1, K2 = np.meshgrid(k1v, k2v, indexing='ij')
for i in range(Nk):
    for j in range(Nk):
        k_phys = k1v[i]*HAL_B1 + k2v[j]*HAL_B2
        _, v = np.linalg.eigh(haldane_hamiltonian(k_phys, t1=t1_H, t2=t2_H, phi=phi_H))
        evecs_H_2d[i, j] = v

C_H = chern_number(evecs_H_2d, flat_bands=[0])
Omega_H = berry_curvature_grid(evecs_H_2d, flat_bands=[0], dk1=1/Nk, dk2=1/Nk)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pcm = axes[0].pcolormesh(K1, K2, Omega_H, cmap='RdBu_r', shading='auto')
plt.colorbar(pcm, ax=axes[0], label='$\\Omega(\\mathbf{k})$')
axes[0].set_title(f'Haldane: Berry Curvature  $C={C_H:.2f}$')
axes[0].set_xlabel('$k_1$ (frac.)'); axes[0].set_ylabel('$k_2$ (frac.)')

# Quantum metric trace Tr[g] vs |Ω|
P_H_flat = projection_matrix_from_vecs(evecs_H_2d.reshape(Nk*Nk, 2, 2), [0]).reshape(Nk, Nk, 2, 2)
dP1 = np.gradient(P_H_flat, 1/Nk, axis=0)
dP2 = np.gradient(P_H_flat, 1/Nk, axis=1)
# Tr[g] = Re Tr[(∂1P)(∂2P) - (∂2P)(∂1P)] / 2 ... simplified: Tr[g] = Tr[∂1P ∂1P + ∂2P ∂2P] / 2
# More directly: metric trace via g_{μν} = Re tr[(1-P) ∂_μP ∂_νP]
g11 = np.einsum('...ij,...jk,...ki->...', (np.eye(2)-P_H_flat), dP1, dP1).real
g22 = np.einsum('...ij,...jk,...ki->...', (np.eye(2)-P_H_flat), dP2, dP2).real
Tr_g_H = g11 + g22  # Tr[g]
deviation_H = Tr_g_H - np.abs(Omega_H)  # should be ≥ 0; = 0 is "ideal geometry"

axes[1].pcolormesh(K1, K2, deviation_H, cmap='hot_r', shading='auto', vmin=0)
pcm2 = axes[1].pcolormesh(K1, K2, deviation_H, cmap='hot_r', shading='auto', vmin=0)
plt.colorbar(pcm2, ax=axes[1], label='$\\mathrm{Tr}[g]-|\\Omega|$')
axes[1].set_title('Haldane: Deviation from Ideal Geometry')
axes[1].set_xlabel('$k_1$ (frac.)'); axes[1].set_ylabel('$k_2$ (frac.)')
plt.tight_layout()
plt.savefig('../docs/haldane_berry.png', dpi=110)
plt.show()

T_H = np.mean(deviation_H)
print(f'Chern number C = {C_H:.3f}')
print(f'Mean Tr[g]-|Ω| (ideality deviation T) = {T_H:.4f}')
print(f'  (T≈0 ↔ ideal geometry, favors FCI; T>>0 ↔ CDW tendency)')

## 3. 模型 B — QWZ 方格晶格 Chern 绝缘体

**哈密顿量**（Qi–Wu–Zhang 2006；Tešanović 1989 类型，综述 §2.3）：
$$H(\mathbf{k}) = \sin k_x\,\sigma_x + \sin k_y\,\sigma_y + (2 - m - \cos k_x - \cos k_y)\,\sigma_z$$

Chern 数：$C=+1$（$0 < m < 2$），$C=-1$（$2 < m < 4$），$C=0$（其他）。

这是最简单的方格晶格拓扑绝缘体，与 Tešanović (1989) 磁通相模型同属非本征 Chern 绝缘体。综述 §2.3 讨论了此类模型在零外磁场下自发出现量子反常霍尔效应（$\sigma_{xy}=e^2/h$）的机制。取 $m=1$（$C=+1$ 相）作为基准。

In [ ]:
# ── QWZ band structure ────────────────────────────────────────────────────────
# BZ: kx,ky ∈ (-π, π]. High-symmetry points in Cartesian.
Gamma_Q = np.array([0.0, 0.0])
X_Q     = np.array([np.pi, 0.0])
M_Q     = np.array([np.pi, np.pi])

k_path_Q = make_path([Gamma_Q, X_Q, M_Q, Gamma_Q])
k_dist_Q = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(k_path_Q, axis=0), axis=1))])
tick_pos_Q = [0,
              np.linalg.norm(X_Q),
              np.linalg.norm(X_Q) + np.linalg.norm(M_Q - X_Q),
              k_dist_Q[-1]]

# Band structure for different m values
fig, ax = plt.subplots(figsize=(6, 4))
colors = plt.cm.viridis(np.linspace(0.1, 0.85, 4))
m_vals = [0.5, 1.0, 1.5, 2.5]
for i, m_v in enumerate(m_vals):
    Es = np.array([np.linalg.eigvalsh(qwz_hamiltonian(k, m=m_v)) for k in k_path_Q])
    label = f'$m={m_v}$, C={"±1" if 0<m_v<2 or 2<m_v<4 else "0"}'
    for n in range(2):
        ax.plot(k_dist_Q, Es[:, n], color=colors[i], lw=1.5,
                label=label if n == 0 else None)

for xp in tick_pos_Q:
    ax.axvline(xp, color='gray', lw=0.5, ls='--')
ax.set_xticks(tick_pos_Q)
ax.set_xticklabels(['Γ', 'X', 'M', 'Γ'])
ax.set_ylabel('$E$')
ax.set_title('QWZ Square-Lattice Chern Insulator Band Structure')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/qwz_bands.png', dpi=110)
plt.show()

# Bandwidth and gap for m=1.0
m_QWZ = 1.0
Es_default_Q = np.array([np.linalg.eigvalsh(qwz_hamiltonian(k, m=m_QWZ)) for k in k_path_Q])
W_Q   = Es_default_Q[:, 0].max() - Es_default_Q[:, 0].min()
gap_Q = Es_default_Q[:, 1].min() - Es_default_Q[:, 0].max()
print(f'm = {m_QWZ}: lower-band width W = {W_Q:.4f},  gap Δ = {gap_Q:.4f},  M = Δ/W = {gap_Q/W_Q:.2f}')
print(f'(Wide dispersive band — not a flat-band model)')

In [ ]:
# ── Berry curvature & Chern number (QWZ) ────────────────────────────────────
Nk = 40
k1v = np.linspace(0, 1, Nk, endpoint=False)
k2v = np.linspace(0, 1, Nk, endpoint=False)
K1, K2 = np.meshgrid(k1v, k2v, indexing='ij')

evecs_Q_2d = np.zeros((Nk, Nk, 2, 2), dtype=complex)
for i in range(Nk):
    for j in range(Nk):
        # BZ: kx ∈ (-π, π], parameterized as frac × 2π shifted to (-π, π]
        k_phys = np.array([(k1v[i] - 0.5) * 2 * np.pi,
                            (k2v[j] - 0.5) * 2 * np.pi])
        _, v = np.linalg.eigh(qwz_hamiltonian(k_phys, m=m_QWZ))
        evecs_Q_2d[i, j] = v

C_Q    = chern_number(evecs_Q_2d, flat_bands=[0])
Omega_Q = berry_curvature_grid(evecs_Q_2d, flat_bands=[0], dk1=1/Nk, dk2=1/Nk)

P_Q_flat = projection_matrix_from_vecs(evecs_Q_2d.reshape(Nk*Nk, 2, 2), [0]).reshape(Nk, Nk, 2, 2)
dP1_Q = np.gradient(P_Q_flat, 1/Nk, axis=0)
dP2_Q = np.gradient(P_Q_flat, 1/Nk, axis=1)
g11_Q = np.einsum('...ij,...jk,...ki->...', (np.eye(2)-P_Q_flat), dP1_Q, dP1_Q).real
g22_Q = np.einsum('...ij,...jk,...ki->...', (np.eye(2)-P_Q_flat), dP2_Q, dP2_Q).real
Tr_g_Q = g11_Q + g22_Q
deviation_Q = Tr_g_Q - np.abs(Omega_Q)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pcm = axes[0].pcolormesh(K1, K2, Omega_Q, cmap='RdBu_r', shading='auto')
plt.colorbar(pcm, ax=axes[0], label='$\\Omega(\\mathbf{k})$')
axes[0].set_title(f'QWZ: Berry Curvature  $C={C_Q:.2f}$')
axes[0].set_xlabel('$k_1$ (frac.)'); axes[0].set_ylabel('$k_2$ (frac.)')

pcm2 = axes[1].pcolormesh(K1, K2, deviation_Q, cmap='hot_r', shading='auto', vmin=0)
plt.colorbar(pcm2, ax=axes[1], label='$\\mathrm{Tr}[g]-|\\Omega|$')
axes[1].set_title('QWZ: Deviation from Ideal Geometry')
axes[1].set_xlabel('$k_1$ (frac.)'); axes[1].set_ylabel('$k_2$ (frac.)')
plt.tight_layout()
plt.savefig('../docs/qwz_berry.png', dpi=110)
plt.show()

T_Q = np.mean(deviation_Q)
print(f'Chern number C = {C_Q:.3f}  (convention: −1 → C=+1 phase)')
print(f'Mean Tr[g]-|Ω| (ideality deviation T) = {T_Q:.4f}')
print(f'  (Dispersive band has large T — not geometrically ideal for FCI)')

## 4. QGN 嵌套图

对每个模型沿 1D $k$-路径（沿倒格子方向）计算嵌套参数
$$\tilde{\omega}_0^Q = \lambda_{\min}(\Pi^Q) \geq 0$$

$\tilde{\omega}_0^Q = 0$ 对应**完美量子几何嵌套（QGN）**，意味着在波矢 $\mathbf{Q}$ 处系统对 CDW（ph 道）或配对（pp 道）不稳定性无几何阻力。

我们比较三个模型在 pp 道（Cooper 对不稳定性）和 ph 道（Peierls/CDW 不稳定性）的嵌套图。

In [ ]:
# ── Helper: compute nestability along a 1D k-strip ───────────────────────────
def compute_nestability_1d(H_func, k_strip, Q_grid, band_idx=0, verbose=True):
    """Nestability map for a 2D model along a 1D k-strip.

    k_strip: (Nk, 2) Cartesian k-points uniformly spaced along some direction
    Q_grid:  (NQ,) scalar Q values (magnitude along the same direction)
    Returns: (omega_pp, omega_ph), each shape (NQ,)
    """
    Nk = len(k_strip)
    _, evecs = diagonalize_model(H_func, k_strip)
    P   = projection_matrix_from_vecs(evecs, flat_bands=[band_idx])
    Q_m = np.eye(evecs.shape[-1])[None, ...] - P

    # Build 1D k_grid as scalar arc-lengths
    diffs = np.linalg.norm(np.diff(k_strip, axis=0), axis=1)
    k_grid_1d = np.concatenate([[0], np.cumsum(diffs)])

    if verbose:
        print('Computing pp...', end=' ', flush=True)
    omega_pp = nestability_map(P, Q_m, k_grid_1d, Q_grid, channel='pp')
    if verbose:
        print('ph...', end=' ', flush=True)
    omega_ph = nestability_map(P, Q_m, k_grid_1d, Q_grid, channel='ph')
    if verbose:
        print('done.')
    return omega_pp, omega_ph

In [ ]:
# ── Haldane: QGN along Γ→K direction ────────────────────────────────────────
Nk1d = 60
NQ   = 40

# k-strip along HAL_B1 direction
k_frac_H_strip = np.linspace(0, 1, Nk1d, endpoint=False)
k_strip_H = np.outer(k_frac_H_strip, HAL_B1)     # (Nk1d, 2) Cartesian

# Q grid: magnitude along HAL_B1, from 0 to |HAL_B1|
Q_frac_H = np.linspace(0, 1, NQ, endpoint=False)
Q_mag_H  = Q_frac_H * np.linalg.norm(HAL_B1)

print('Haldane nestability:')
omega_pp_H, omega_ph_H = compute_nestability_1d(
    lambda k: haldane_hamiltonian(k, t1=t1_H, t2=t2_H, phi=phi_H),
    k_strip_H, Q_mag_H, band_idx=0
)

In [ ]:
# ── QWZ: QGN along Γ→X direction ────────────────────────────────────────────
# BZ kx ∈ (-π, π). Strip along kx from -π to π (= along QWZ_B1 direction)
Nk1d = 60
NQ   = 40
k_frac_Q_strip = np.linspace(0, 1, Nk1d, endpoint=False)

k_strip_Q = np.column_stack([(k_frac_Q_strip - 0.5) * 2 * np.pi,  # kx: -π → π
                               np.zeros(Nk1d)])                      # ky = 0

# Arc-length along strip (scalar)
Q_mag_Q = np.linspace(0, 2 * np.pi, NQ, endpoint=False)  # Q from 0 to 2π (= |QWZ_B1|)

print('QWZ nestability:')
omega_pp_Q, omega_ph_Q = compute_nestability_1d(
    lambda k: qwz_hamiltonian(k, m=m_QWZ),
    k_strip_Q, Q_mag_Q, band_idx=0
)
Q_frac_Q = Q_mag_Q / (2 * np.pi)  # normalised by |B1|

In [ ]:
# ── Triangular FCI: QGN along B1 direction ───────────────────────────────────
_FCI_B1 = np.array([4*np.pi/3, 0.0])
_FCI_B2 = np.array([-2*np.pi/3, 2*np.pi/np.sqrt(3)])

k_strip_TRI = np.outer(np.linspace(0, 1, Nk1d, endpoint=False), _FCI_B1)
Q_frac_TRI  = np.linspace(0, 1, NQ, endpoint=False)
Q_mag_TRI   = Q_frac_TRI * np.linalg.norm(_FCI_B1)

print('Triangular FCI nestability (t\'/t=0.2):')
omega_pp_TRI, omega_ph_TRI = compute_nestability_1d(
    lambda k: fci_triangular_hamiltonian(k, t=1.0, tp=0.2),
    k_strip_TRI, Q_mag_TRI, band_idx=0
)

In [ ]:
# ── Plot comparison ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (omega_H, omega_Qwz, omega_TRI), label in zip(
        axes,
        [(omega_pp_H, omega_pp_Q, omega_pp_TRI),
         (omega_ph_H, omega_ph_Q, omega_ph_TRI)],
        ['p-p (Cooper)', 'p-h (Peierls/CDW)']):

    ax.plot(Q_frac_H,   omega_H,   'C0-o', ms=4, lw=1.5, label='Haldane honeycomb')
    ax.plot(Q_frac_Q,   omega_Qwz, 'C1-s', ms=4, lw=1.5, label='QWZ square lattice')
    ax.plot(Q_frac_TRI, omega_TRI, 'C2-^', ms=4, lw=1.5, label='Triangular FCI')
    ax.set_xlabel('$Q / |\\mathbf{b}_1|$')
    ax.set_ylabel('$\\tilde{\\omega}_0^Q$')
    ax.set_title(f'QGN Nestability  ({label})')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)

plt.suptitle('QGN Nestability Map Comparison: Three Chern-band Models', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('../docs/qgn_comparison.png', dpi=110, bbox_inches='tight')
plt.show()

print('\n=== QGN Summary ===')
for name, op, oh, qf in [
    ('Haldane',        omega_pp_H, omega_ph_H, Q_frac_H),
    ('QWZ',            omega_pp_Q, omega_ph_Q, Q_frac_Q),
    ('Triangular FCI', omega_pp_TRI, omega_ph_TRI, Q_frac_TRI),
]:
    print(f'{name:16s}  pp_min={op.min():.4f} @ Q/b1={qf[np.argmin(op)]:.3f}'
          f'   ph_min={oh.min():.4f} @ Q/b1={qf[np.argmin(oh)]:.3f}')

## 5. QWZ：Q=(π,0) 处的嵌套矩阵

QWZ 模型 BZ 的特殊波矢为 $\mathbf{Q} = (\pi, 0)$（X 点）。对于正方晶格，这正是 $C_4$ 对称性允许的 CDW 波矢之一（另一个为 $(0,\pi)$，等价）。此处提取 $Q=\pi$ 处 $\Pi^Q$ 的零空间，得到**嵌套矩阵** $N^Q$，刻画主导不稳定性的轨道结构。

In [ ]:
# ── Nesting matrix at Q = π (X-point CDW wave vector) ────────────────────────
# Build k_strip along kx with ky=0
_, evecs_1d_Q = diagonalize_model(lambda k: qwz_hamiltonian(k, m=m_QWZ), k_strip_Q)
P_1d_Q  = projection_matrix_from_vecs(evecs_1d_Q, flat_bands=[0])
Qm_1d_Q = np.eye(2)[None, ...] - P_1d_Q

# Arc-lengths for k_strip_Q
k_1d_Q = np.abs(k_strip_Q[:, 0])   # |kx| arc from 0 to π (symmetric strip)
# More robustly, use cumulative arc-length:
k_1d_Q = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(k_strip_Q, axis=0), axis=1))])

Q_X = np.pi   # Q = π

print('ph channel at Q = π (X-point):')
N_ph_Q, omega_ph_X = nesting_matrix(P_1d_Q, Qm_1d_Q, k_1d_Q, Q_X, channel='ph', tol=1e-2)
print(f'  ω_min = {omega_ph_X:.6f}')
if N_ph_Q:
    print(f'  # null vectors = {len(N_ph_Q)}')
    for i, N in enumerate(N_ph_Q):
        print(f'  N_{i} =\n{N.round(4)}')
else:
    print('  No near-zero eigenvalues (threshold 1e-2)')

print(f'\npp channel at Q = π:')
N_pp_Q, omega_pp_X = nesting_matrix(P_1d_Q, Qm_1d_Q, k_1d_Q, Q_X, channel='pp', tol=1e-2)
print(f'  ω_min = {omega_pp_X:.6f}')

## 6. 量子几何理想性与 FCI 稳定性：三模型比较

Trung & Yang (2021)（综述 §3.3）给出判据：FCI 稳定 ↔ $\mathcal{T} = \langle\mathrm{Tr}[g] - |\Omega|\rangle \approx 0$（理想量子几何）。
CDW 倾向 ↔ $\mathcal{T} \gg 0$（远离理想几何）。

下面对三个模型系统地计算 $\mathcal{T}$ 并与 QGN 嵌套参数对比。

In [ ]:
# ── Triangular FCI: ideality deviation ──────────────────────────────────────
_FCI_B1 = np.array([4*np.pi/3, 0.0])
_FCI_B2 = np.array([-2*np.pi/3, 2*np.pi/np.sqrt(3)])

evecs_TRI_2d = np.zeros((Nk, Nk, 2, 2), dtype=complex)
for i in range(Nk):
    for j in range(Nk):
        k_phys = k1v[i]*_FCI_B1 + k2v[j]*_FCI_B2
        _, v = np.linalg.eigh(fci_triangular_hamiltonian(k_phys, t=1.0, tp=0.2))
        evecs_TRI_2d[i, j] = v

C_TRI    = chern_number(evecs_TRI_2d, flat_bands=[0])
Omega_TRI = berry_curvature_grid(evecs_TRI_2d, flat_bands=[0], dk1=1/Nk, dk2=1/Nk)
P_TRI_flat = projection_matrix_from_vecs(evecs_TRI_2d.reshape(Nk*Nk, 2, 2), [0]).reshape(Nk, Nk, 2, 2)
dP1_TRI = np.gradient(P_TRI_flat, 1/Nk, axis=0)
dP2_TRI = np.gradient(P_TRI_flat, 1/Nk, axis=1)
g11_TRI = np.einsum('...ij,...jk,...ki->...', (np.eye(2)-P_TRI_flat), dP1_TRI, dP1_TRI).real
g22_TRI = np.einsum('...ij,...jk,...ki->...', (np.eye(2)-P_TRI_flat), dP2_TRI, dP2_TRI).real
Tr_g_TRI = g11_TRI + g22_TRI
deviation_TRI = Tr_g_TRI - np.abs(Omega_TRI)
T_TRI = np.mean(deviation_TRI)

print(f'Triangular FCI: C={C_TRI:.3f},  T={T_TRI:.4f}')

# ── Summary table ────────────────────────────────────────────────────────────
print('\n=== Quantum Geometry Summary ===')
print(f'{"Model":<22} {"C":>6} {"W":>8} {"Δ":>8} {"M=Δ/W":>8} {"T=⟨Tr[g]-|Ω|⟩":>16}')
print('-'*72)

Es_TRI_path = np.array([np.linalg.eigvalsh(fci_triangular_hamiltonian(k, t=1.0, tp=0.2))
                         for k in make_path([np.zeros(2), _FCI_B1, _FCI_B2, np.zeros(2)])])
W_TRI  = Es_TRI_path[:, 0].max() - Es_TRI_path[:, 0].min()
gap_TRI = Es_TRI_path[:, 1].min() - Es_TRI_path[:, 0].max()

for name, C, W, gap, T in [
    ('Haldane (t2=0.3)',   C_H,   W_H,   gap_H,   T_H),
    ('QWZ (m=1.0)',        C_Q,   W_Q,   gap_Q,   T_Q),
    ('Triangular FCI',     C_TRI, W_TRI, gap_TRI, T_TRI),
]:
    print(f'{name:<22} {C:>6.2f} {W:>8.3f} {gap:>8.3f} {gap/max(W,1e-6):>8.1f} {T:>16.4f}')

In [ ]:
# ── Final summary figure ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Row 0: Berry curvature
ax00 = fig.add_subplot(gs[0, 0])
ax01 = fig.add_subplot(gs[0, 1])
ax02 = fig.add_subplot(gs[0, 2])
for ax, Om, title in [(ax00, Omega_H,   f'Haldane  $C={C_H:.0f}$'),
                       (ax01, Omega_Q,   f'QWZ  $C={C_Q:.0f}$'),
                       (ax02, Omega_TRI, f'Triangular FCI  $C={C_TRI:.0f}$')]:
    pcm = ax.pcolormesh(K1, K2, Om, cmap='RdBu_r', shading='auto')
    plt.colorbar(pcm, ax=ax, shrink=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('$k_1$'); ax.set_ylabel('$k_2$')

# Row 1: QGN nestability (ph channel)
ax10 = fig.add_subplot(gs[1, :])
ax10.plot(Q_frac_H,   omega_ph_H,   'C0-o', ms=4, lw=2, label='Haldane honeycomb')
ax10.plot(Q_frac_Q,   omega_ph_Q,   'C1-s', ms=4, lw=2, label='QWZ square lattice')
ax10.plot(Q_frac_TRI, omega_ph_TRI, 'C2-^', ms=4, lw=2, label='Triangular FCI')
ax10.axvline(0.5, color='gray', ls='--', lw=1, label='$Q=\\pi$ (X-point)')
ax10.set_xlabel('$Q / |\\mathbf{b}_1|$', fontsize=12)
ax10.set_ylabel('$\\tilde{\\omega}_0^Q$', fontsize=12)
ax10.set_title('p-h Channel Nestability (CDW sensitivity)', fontsize=12)
ax10.set_xlim(0, 1)
ax10.legend(fontsize=10)

plt.suptitle('QGN Analysis: Topology–CDW Coexistence Models', fontsize=14, y=1.01)
plt.savefig('../docs/summary_topology_CDW.png', dpi=110, bbox_inches='tight')
plt.show()

## 7. 物理解读

### 量子几何理想性 ($\mathcal{T}$) 与 QGN 的对应

| 模型 | 能带平坦性 $M$ | $\mathcal{T}$ | 物理含义 |
|------|--------------|--------------|---------|
| Haldane 蜂窝 | 低（非平带）| 较大 | 色散带，Berry 曲率非均匀，CDW 几何阻力弱 |
| QWZ 方格 | 低（非平带）| 中等 | 最简 Chern 绝缘体，Berry 曲率集中于 Γ 点 |
| 三角晶格 FCI | 高（近平带）| 小 | 近理想几何，FCI 稳定，CDW 波矢依赖几何 |

### 与综述文章的联系

**§2.3 Tešanović 磁通相 / QWZ**：两者均为 $C=\pm1$ 方格晶格 2 能带模型，在无外磁场下自发产生 $\sigma_{xy}=e^2/h$（量子反常霍尔效应）。QWZ 更接近连续极限，因而 Berry 曲率集中于 $\Gamma$ 附近——QGN 在小 $Q$ 处嵌套最强。

**§3.2 Neupert et al. (2011) Haldane+$V$**：在 Haldane 模型基础上加近邻相互作用 $V$，$\nu=1/3$ 处实现 FCI。非平带使得 $\mathcal{T}$ 较大，但 QGN 仍可识别出几何偏好的 CDW 波矢。

**§4.1 Kourtis–Daghofer 共存态**：文章发现 $C=+1$ 平带（类似三角格子 FCI）在 $\nu=1/3$ 时 FCI 与 CDW 共存，简并度 $3_{\rm FCI}\times 2_{\rm CDW}=6$。三角格子 FCI 的 $\mathcal{T}$ 最小——近理想几何支持 FCI，QGN 在特定 $Q$ 处的极小暗示 CDW 不稳定性的波矢选择。

### QGN 框架的局限性

QGN 为**单粒子几何**分析，不含相互作用强度信息。实际 FCI vs CDW 竞争由几何（QGN/理想性）和相互作用（$V_1, V_3$）共同决定。QGN 极小处的 CDW 是否在真实相图中实现，还需 ED/DMRG 验证。